# Exploración guiada (punto de partida, Hito 1)

**Responsable:** Leo (Rol D, con apoyo de todos)  
**Revisor:** Dan (Rol B)

## Qué queda resuelto

- [x] Cargar `01-22-2020.csv`: primeras filas, número de filas y columnas, tipos de datos
- [x] Contar los nulos por columna y explicar qué significan aquí
- [x] Nombres a snake_case y fecha de actualización en AAAA-MM-DD
- [x] Unir los reportes de enero a junio de 2020 (dos cambios de esquema)
- [x] Top 10 de países al 30-jun-2020 y evolución de Chile (gráfico de líneas)
- [x] Perfilado automático (ydata-profiling) y tres observaciones

In [ ]:
import sys, pathlib
RAIZ = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(RAIZ))
import pandas as pd
import matplotlib.pyplot as plt

RAW_DAILY = RAIZ / 'data' / 'raw' / 'daily_reports'
RENOMBRAR = {
    'Province/State': 'province_state', 'Province_State': 'province_state',
    'Country/Region': 'country_region', 'Country_Region': 'country_region',
    'Last Update': 'last_update', 'Last_Update': 'last_update',
    'Latitude': 'lat', 'Lat': 'lat', 'Longitude': 'long', 'Long_': 'long',
    'Incidence_Rate': 'incident_rate', 'Incident_Rate': 'incident_rate',
    'Case-Fatality_Ratio': 'case_fatality_ratio', 'Case_Fatality_Ratio': 'case_fatality_ratio',
}

## Cargar `01-22-2020.csv`: primeras filas, número de filas y columnas, tipos de datos

In [ ]:
df_ini = pd.read_csv(RAW_DAILY / '01-22-2020.csv')
print('Dimensiones:', df_ini.shape)
print(df_ini.dtypes)
display(df_ini.head())

## Contar los nulos por columna y explicar qué significan aquí

In [ ]:
print(df_ini.isna().sum())
# En 01-22-2020.csv, los nulos en Province/State indican paises reportados a nivel nacional;
# los nulos en Confirmed, Deaths o Recovered equivalen a cero casos reportados al inicio del brote.

## Nombres a snake_case y fecha de actualización en AAAA-MM-DD

In [ ]:
df_std = df_ini.rename(columns=RENOMBRAR)
df_std.columns = df_std.columns.str.lower()
df_std['last_update_iso'] = pd.to_datetime(df_std['last_update']).dt.strftime('%Y-%m-%d')
display(df_std.head())

## Unir los reportes de enero a junio de 2020 (dos cambios de esquema)

In [ ]:
dfs_sem1 = []
for fecha in pd.date_range('2020-01-22', '2020-06-30', freq='D'):
    ruta = RAW_DAILY / f"{fecha.strftime('%m-%d-%Y')}.csv"
    if not ruta.exists():
        continue
    d = pd.read_csv(ruta, low_memory=False).rename(columns=RENOMBRAR)
    d.columns = d.columns.str.lower()
    d['fecha_reporte'] = fecha.strftime('%Y-%m-%d')
    if 'country_region' in d.columns:
        d['country_region'] = d['country_region'].replace({'Mainland China': 'China'})
    dfs_sem1.append(d)
df_sem1 = pd.concat(dfs_sem1, ignore_index=True)
print('Filas ene-jun 2020:', len(df_sem1), '| Columnas unificadas:', list(df_sem1.columns))

## Top 10 de países al 30-jun-2020 y evolución de Chile (gráfico de líneas)

In [ ]:
top10_jun = (df_sem1[df_sem1['fecha_reporte'] == '2020-06-30']
             .groupby('country_region', as_index=False)['confirmed'].sum()
             .sort_values('confirmed', ascending=False).head(10))
display(top10_jun)

cl_sem1 = (df_sem1[df_sem1['country_region'] == 'Chile']
           .groupby('fecha_reporte', as_index=False)['confirmed'].sum())
cl_sem1['fecha_reporte'] = pd.to_datetime(cl_sem1['fecha_reporte'])
ax = cl_sem1.plot(x='fecha_reporte', y='confirmed', figsize=(9, 4), legend=False, title='Chile · Casos confirmados acumulados (ene-jun 2020)')
ax.set_ylabel('Confirmados acumulados'); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## Perfilado automático (ydata-profiling) y tres observaciones

1. **Cambios de esquema progresivos:** entre enero y junio de 2020 pasan de 6 columnas (Etapa 1) a 14 columnas (Etapa 4), introduciendo coordenadas (`lat`, `long`), granularidad subnacional (`fips`, `admin2`) e indicadores epidemiológicos.
2. **Asimetría extrema de valores acumulados:** unas pocas entidades (EE. UU., Brasil, Rusia, regiones densas) concentran la mayoría de los casos mientras la mediana territorial es baja.
3. **Cambio de granularidad en Chile:** hasta mayo de 2020 Chile reporta 1 fila nacional por día y desde fines de mayo se desagrega en 16 regiones más `Unknown`.

In [ ]:
from ydata_profiling import ProfileReport
os_dir = RAIZ / 'reports' / 'profiling'
os_dir.mkdir(parents=True, exist_ok=True)
muestra = df_sem1.sample(min(20000, len(df_sem1)), random_state=42)
perfil = ProfileReport(muestra, title='Perfilado COVID-19 (ene-jun 2020)', minimal=True)
perfil.to_file(os_dir / 'perfil_ene_jun_2020.html')
print('Perfilado guardado en reports/profiling/perfil_ene_jun_2020.html')